# NB06 · 干预迭代：预测 → 干预 → 实测

| | |
|---|---|
| **目标** | 完成一次科学家式的迭代：先书面预测效果并锁死，再动手，最后解释预测与实测的差 |
| **前置** | NB05 的归因结论 |
| **预计耗时** | 半天人工 + 一次训练的机器时 |
| **产出物** | `results/NB06.json`（含锁定的预测和实测对比） |
| **通过标准** | 预测在动手前已落盘；实测后差值有解释 |

规则：从上到下顺序执行；每个 ✅ 检查点必须核对；最后的复盘必须填写并 commit。


### 干预菜单（只选一个——变量隔离是底线）

1. **补数据**：按 NB05 结论定向采/选 20 条 episode 加入训练集
2. **action chunking**：改 chunk size / action horizon
3. **数据增强**：图像增强或状态噪声
4. **评测分布**：如果 NB05 发现 protocol 问题，修协议（这也算干预，且常常是性价比最高的）
5. **训练预算**：steps 翻倍（最无聊但最常见的对照组）


In [ ]:
import nbutils

baseline = nbutils.latest("NB02")["success_rate"]
PREDICTION = {
    "intervention": "",           # 用一句话描述你选的干预
    "baseline": baseline,
    "predicted": None,            # 预测干预后的 success rate
    "mechanism": "",              # 为什么会有这个效果？一句话机理（必须基于 NB05 的失败分布）
    "prediction_locked": True,
}
assert PREDICTION["intervention"] and PREDICTION["predicted"] is not None
nbutils.log_result("NB06", PREDICTION)   # ⚠️ 先落盘再动手。改预测 = 作弊，账本会留下记录。
print(f"预测已锁定: {PREDICTION['baseline']:.1%} -> {PREDICTION['predicted']:.1%}")

In [ ]:
# 实施干预并重训（复用 NB02 的命令模板，只改你选定的那一个变量），
# 然后用 NB03 协议重新评测。全部跑完后填实测值：
MEASURED = None    # 干预后的 success rate
assert MEASURED is not None
n_eval = nbutils.latest("NB03")["n_episodes"]
lo, hi = nbutils.wilson_ci(int(MEASURED * n_eval), n_eval)
b_lo, b_hi = nbutils.wilson_ci(int(baseline * n_eval), n_eval)
overlap = not (lo > b_hi or hi < b_lo)
print(f"baseline {baseline:.1%} [{b_lo:.1%},{b_hi:.1%}]  →  {MEASURED:.1%} [{lo:.1%},{hi:.1%}]")
print("CI 重叠 → 无法区分" if overlap else "CI 分离 → 差异显著")
nbutils.log_result("NB06", {"measured": MEASURED, "ci": [lo, hi], "significant": not overlap})

## 分析：预测与实测的差从哪来（三选一，给证据）

1. **机理对，幅度错**：方向猜对了但高/低估了——你对「一个失败模式占比 → 修复它的收益」的换算哪里错了？
2. **机理错**：干预没作用在你以为的失败模式上——重看 NB05 的视频，找反例。
3. **测量噪声**：差值在 CI 内——那这次实验的功效够吗？（用 NB03 实验三回答该用多大的 n 重测。）

> 不在乎涨没涨，在乎差值有没有解释。这一页《预测→干预→实测》就是你未来跟任何团队证明自己是科学家的最小样本。


## 复盘（必填，不填不算完成这本 notebook）

> 复盘写在这里并 commit。允许粗糙，禁止事后美化。

- **预期 vs 实际**：
- **最大的一个意外**：
- **卡最久的一步和根因**：
- **用一句话向非技术人解释本次学到的东西**：
- **进入下一本之前要做的一个动作**：
